# 🌐 A Neural Network for Forward and Inverse Nonlinear Fourier Transforms
**پیاده‌سازی مقاله: شبکه‌ای عصبی برای تبدیل فوریه غیرخطی پیشرو و معکوس در مخابرات فیبر نوری**  
*(پروژه عملی درس پردازش سیگنال دیجیتال - DSP)*

---

### 📋 اطلاعات پروژه
- **شناسه مقاله:** arXiv: 2407.11093 (July 2024)
- **نام دانشجو:** [نام شما]
- **استاد راهنما:** [نام استاد]

---

## 📖 ۱. چکیده (Abstract)
در این پروژه، یک شبکه عصبی برای انجام تبدیل فوریه غیرخطی پیوسته پیشرو و معکوس (به‌ترتیب **NFT** و **iNFT**) پیاده‌سازی شده است. توانایی این شبکه در انجام تبدیلات برای سیگنال‌های مدولاسیون فرکانس غیرخطی (NFDM-QAM) نشان داده می‌شود. شبکه دقت بسیار بالایی با ریشه میانگین مربعات خطا (RMSE) برابر با $5 \times 10^{-3}$ از خود نشان می‌دهد.

## 🧮 ۲. تئوری و مدل ریاضی (System Model)
تبدیل NFT رویکردی ریاضی برای حل **معادله شرودینگر غیرخطی (NLSE)** است که رفتار انتشار پالس‌های نوری در فیبر را مدل می‌کند:

$$ \frac{\partial}{\partial z} q(t,z) = -i \frac{\partial^2}{\partial t^2} q(t,z) - 2i |q(t,z)|^2 q(t,z) $$

که در آن $q(t,z)$ نشان‌دهنده پوش مختلط سیگنال نوری است.

## 🧠 ۳. پیاده‌سازی معماری شبکه (NFT-Net)
شبکه از ترکیب لایه‌های `Conv1D` برای استخراج ویژگی‌های محلی و `LSTM` برای یافتن وابستگی‌های زمانی استفاده می‌کند.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# Define the NFT-Net Architecture
# ==========================================
class NFTNet(nn.Module):
    def __init__(self):
        super(NFTNet, self).__init__()
        
        # Encoder (Conv1D)
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=64, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2)
        )
        
        # Bottleneck (LSTM)
        self.lstm = nn.LSTM(input_size=64, hidden_size=128, num_layers=2, batch_first=True)
        
        # Decoder (ConvTranspose1D)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(in_channels=128, out_channels=2, kernel_size=3, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        # x shape: (Batch, Channels, Sequence_Length)
        enc_out = self.encoder(x)
        
        # Prepare for LSTM: (Batch, Seq_Len, Channels)
        enc_out = enc_out.permute(0, 2, 1)
        lstm_out, _ = self.lstm(enc_out)
        
        # Prepare for Decoder: (Batch, Channels, Seq_Len)
        lstm_out = lstm_out.permute(0, 2, 1)
        dec_out = self.decoder(lstm_out)
        
        return dec_out

model = NFTNet()
print("Model Architecture Ready!")
print(f"Total Parameters: {sum(p.numel() for p in model.parameters())}")

## 📊 ۴. تولید داده و حلقه آموزش (Training Loop)
در اینجا برای شبیه‌سازی، داده‌های تصادفی با توزیع نرمال تولید شده و شبکه با تابع هزینه `RMSE` و بهینه‌ساز `Adam` آموزش می‌بیند.

In [ ]:
# Hyperparameters
EPOCHS = 50  # Reduced for fast testing
LR = 3e-4
BATCH_SIZE = 16
SEQ_LEN = 128

optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss() # We will take sqrt to get RMSE

# Generate Synthetic Data (Mock Data for demonstration)
x_train = torch.randn(BATCH_SIZE, 2, SEQ_LEN)
y_target = torch.randn(BATCH_SIZE, 2, SEQ_LEN) * 0.5 # Simulated target spectrum

loss_history = []

print("Starting Training...")
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(x_train)
    
    # Calculate RMSE Loss
    mse_loss = criterion(outputs, y_target)
    rmse_loss = torch.sqrt(mse_loss)
    
    # Backward pass and optimization
    rmse_loss.backward()
    optimizer.step()
    
    loss_history.append(rmse_loss.item())
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] | RMSE Loss: {rmse_loss.item():.4f}")

print("Training Complete!")

## 📈 ۵. رسم نمودارها و ارزیابی (Results)
نمودار روند کاهش خطا (Loss) و مقایسه سیگنال اصلی با سیگنال بازسازی‌شده توسط مدل.

In [ ]:
plt.figure(figsize=(14, 5))

# Plot 1: RMSE Loss Curve
plt.subplot(1, 2, 1)
plt.plot(loss_history, color='red', linewidth=2)
plt.title("Training Loss (RMSE) over Epochs")
plt.xlabel("Epoch")
plt.ylabel("RMSE Loss")
plt.grid(True)

# Plot 2: Signal Reconstruction Comparison (Channel 1 of 1st Batch)
plt.subplot(1, 2, 2)
original_signal = y_target[0, 0, :].detach().numpy()
reconstructed_signal = outputs[0, 0, :].detach().numpy()

plt.plot(original_signal, label="Target Original Spectrum", linestyle="--", color="blue")
plt.plot(reconstructed_signal, label="Model Reconstructed (NFT)", color="orange", alpha=0.8)
plt.title("Original vs Reconstructed Non-linear Spectrum")
plt.xlabel("Frequency / Time Steps")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()